# 5. Model Submission (Ensemble)

## Objective
Load trained ensemble models and generate predictions using beam voting.
Save predictions to `./submission.csv`

### Approach:
1. Load models **one at a time** to avoid GPU OOM errors
2. Generate translations incrementally
3. Aggregate results as models are processed
4. Create submission file

### Ensemble Models:
1. T5-small v1
2. T5-base
3. MarianMT
4. ByT5
5. T5-small v2

### Memory Management:
Due to GPU memory constraints, we load models sequentially and clear GPU memory after each model.

In [1]:
# Parameters and Paths
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
import json
import re

OUTPUT_DIR = Path("../output")
MODELS_DIR = OUTPUT_DIR / "models"
DATA_DIR = Path("../data")

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda


In [ ]:
def clear_gpu_memory():
    """Clear GPU memory to avoid OOM errors when loading models sequentially."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print(f"GPU memory cleared. Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

def get_gpu_memory_info():
    """Get current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        return f"Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB"
    return "No GPU"

print("GPU memory management functions defined")

In [2]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, MarianTokenizer, MarianMTModel

# Ensemble configuration - models will be loaded one at a time
ENSEMBLE_MODELS = [
    {
        "name": "t5_small_v1",
        "type": "t5",
        "path": "t5_akkadian_translator"
    },
    {
        "name": "t5_base",
        "type": "t5",
        "path": "ensemble/t5_base"
    },
    {
        "name": "marianmt",
        "type": "marian",
        "path": "ensemble/marianmt"
    },
    {
        "name": "byt5",
        "type": "t5",
        "path": "ensemble/byt5"
    },
    {
        "name": "t5_small_v2",
        "type": "t5",
        "path": "ensemble/t5_small_v2"
    }
]

BEAM_SIZE = 5

In [3]:
# Clear initial GPU memory
clear_gpu_memory()
print(f"Initial GPU memory: {get_gpu_memory_info()}")

Initial GPU memory: Allocated: 0.00 GB, Reserved: 0.00 GB


## 5.1 Text Cleaning Function (same as preprocessing)

In [4]:
def clean_transliteration(text):
    """
    Clean Akkadian transliteration - same as in data preparation.
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Replace gaps and breaks first
    text = re.sub(r'\[x\]', '<gap>', text)
    text = re.sub(r'\.{3,}', '<big_gap>', text)
    text = re.sub(r'\[…\s*…\]', '<big_gap>', text)
    text = re.sub(r'\[\s*\]', '', text)
    
    # Remove/modify special characters
    text = re.sub(r'[!?/:.]', '', text)
    text = re.sub(r'[˹˺]', '', text)
    text = re.sub(r'\[|\]', '', text)
    
    # Normalize Ḫ → H and ḫ → h
    text = text.replace('Ḫ', 'H').replace('ḫ', 'h')
    text = text.replace('Ṣ', 'S').replace('ṣ', 's')
    text = text.replace('Ṭ', 'T').replace('ṭ', 't')
    text = text.replace('Š', 'SZ').replace('š', 'sz')
    
    # Clean up extra whitespace
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text

print("Cleaning function defined")

Cleaning function defined


## 5.2 Load Test Data

In [5]:
# Load test data
test_df = pd.read_csv(DATA_DIR / "test.csv")
print(f"Test samples: {len(test_df)}")
print(f"\nTest columns: {test_df.columns.tolist()}")

# Clean test data
test_df['transliteration_cleaned'] = test_df['transliteration'].apply(clean_transliteration)

texts = test_df['transliteration_cleaned'].tolist()
print(f"\nPrepared {len(texts)} texts for translation")

Test samples: 4

Test columns: ['id', 'text_id', 'line_start', 'line_end', 'transliteration']


## 5.3 Model Loading and Translation Functions

In [6]:
def load_model_for_inference(model_info):
    """
    Load a single model for inference.
    Returns the model, tokenizer, and type, or None if model doesn't exist.
    """
    model_path = MODELS_DIR / model_info["path"]
    model_type = model_info["type"]
    model_name = model_info["name"]
    
    if not model_path.exists() or not (model_path / "config.json").exists():
        print(f"  Model not found: {model_name} at {model_path}")
        return None
    
    print(f"  Loading {model_name}...")
    print(f"  GPU memory before: {get_gpu_memory_info()}")
    
    try:
        if model_type == "t5":
            tokenizer = T5Tokenizer.from_pretrained(model_path)
            model = T5ForConditionalGeneration.from_pretrained(model_path)
        elif model_type == "marian":
            tokenizer = MarianTokenizer.from_pretrained(model_path)
            model = MarianMTModel.from_pretrained(model_path)
        else:
            print(f"  Unknown model type: {model_type}")
            return None
        
        model = model.to(device)
        model.eval()
        
        print(f"  GPU memory after: {get_gpu_memory_info()}")
        
        return {
            "model": model,
            "tokenizer": tokenizer,
            "type": model_type,
            "name": model_name
        }
    except Exception as e:
        print(f"  Error loading {model_name}: {e}")
        return None

def unload_model(model_dict):
    """
    Unload a model and clear GPU memory.
    """
    if model_dict is not None:
        del model_dict["model"]
        del model_dict["tokenizer"]
        del model_dict
    clear_gpu_memory()

def translate_with_model(texts, model_info, beam_size=5, batch_size=4):
    """
    Generate translations using a single model.
    Returns a list of translation candidates with scores.
    """
    model_dict = load_model_for_inference(model_info)
    if model_dict is None:
        return [{"text": "", "score": 0, "model": model_info["name"], "beam_rank": 0}] * len(texts)
    
    model = model_dict["model"]
    tokenizer = model_dict["tokenizer"]
    model_type = model_dict["type"]
    model_name = model_dict["name"]
    
    all_candidates = []
    
    for idx in tqdm(range(len(texts)), desc=f"Translating with {model_name}"):
        text = texts[idx]
        candidates = []
        
        try:
            # Prepare input
            if model_type == "t5":
                input_text = "translate Akkadian to English: " + text
            elif model_type == "marian":
                input_text = text
            
            inputs = tokenizer(input_text, return_tensors="pt", 
                              padding=True, truncation=True, max_length=256)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Generate with beam search
            with torch.no_grad():
                outputs = model.generate(
                    input_ids=inputs['input_ids'],
                    attention_mask=inputs['attention_mask'],
                    max_length=128,
                    num_beams=beam_size,
                    num_return_sequences=beam_size,
                    early_stopping=True
                )
            
            # Decode all beams
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            
            # Score candidates
            for beam_idx, translation in enumerate(decoded):
                score = beam_size - beam_idx
                translation = translation.strip()
                if translation:
                    candidates.append({
                        "text": translation,
                        "score": score,
                        "model": model_name,
                        "beam_rank": beam_idx
                    })
        except Exception as e:
            print(f"Error with {model_name} on text {idx}: {e}")
        
        all_candidates.append(candidates if candidates else [{"text": "", "score": 0}])
    
    # Unload model and clear GPU memory
    unload_model(model_dict)
    
    return all_candidates

print("Translation functions defined")

Translation functions defined


## 5.4 Sequential Model Translation

We process models one at a time to avoid GPU OOM. Each model's predictions are aggregated incrementally.

In [ ]:
# Initialize storage for all candidates across models
all_model_candidates = {i: [] for i in range(len(texts))}

available_models = []

print("Checking available models...")
for model_info in ENSEMBLE_MODELS:
    model_path = MODELS_DIR / model_info["path"]
    if model_path.exists() and (model_path / "config.json").exists():
        available_models.append(model_info)
        print(f"  Available: {model_info['name']}")
    else:
        print(f"  Not found: {model_info['name']}")

print(f"\n{len(available_models)} models available for ensemble")

In [ ]:
# Process each model sequentially
for model_idx, model_info in enumerate(available_models):
    print(f"\n{'='*60}")
    print(f"Processing model {model_idx + 1}/{len(available_models)}: {model_info['name']}")
    print(f"{'='*60}")
    
    # Clear GPU before loading new model
    clear_gpu_memory()
    
    # Generate translations with this model
    model_candidates = translate_with_model(texts, model_info, beam_size=BEAM_SIZE)
    
    # Aggregate candidates
    for i, candidates in enumerate(model_candidates):
        all_model_candidates[i].extend(candidates)
    
    print(f"  Total candidates so far: {len(all_model_candidates[0])}")
    
    # Clear GPU after processing this model
    clear_gpu_memory()

## 5.5 Beam Voting - Select Best Translation

In [7]:
def select_best_translation(candidates):
    """
    Select the best translation from a list of candidates using length-normalized scoring.
    """
    if not candidates:
        return ""
    
    best_candidate = None
    best_score = float('-inf')
    
    for cand in candidates:
        text_len = len(cand["text"].split())
        # Length normalization: prefer reasonable length translations
        length_penalty = min(text_len / 10.0, 1.0) if text_len > 0 else 0.1
        final_score = cand["score"] * length_penalty
        
        if final_score > best_score:
            best_score = final_score
            best_candidate = cand
    
    return best_candidate["text"] if best_candidate else ""

# Select best translation for each text
translations = []

print("Selecting best translations using beam voting...")
for i in range(len(texts)):
    candidates = all_model_candidates[i]
    best_translation = select_best_translation(candidates)
    translations.append(best_translation)

print(f"\nSample selection (first 3):")
for i in range(min(3, len(translations))):
    print(f"  Text {i}: Selected '{translations[i][:50]}...'")

Selecting best translations using beam voting...
Sample selection (first 3):
  Text 0: Selected 'From the Kanesh colony to Aur-nya: The Kanesh ...'
  Text 1: Selected 'To the tablet of Ali-ahum, at the disposal of ...'
  Text 2: Selected 'gap> gap> gap> gap> gap> gap> gap> gap> gap> gap>'


## 5.6 Create Submission File

In [8]:
# Create submission dataframe
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'translation': translations
})

# Verify format matches sample submission
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
print("Sample submission format:")
print(sample_submission.head())

print("\nOur submission format:")
print(submission_df.head())

Sample submission format:
   id                                        translation
0   0  Thus  Kanesh, say to the -payers, our messenge...
1   1  In the letter of the City (it is written): Fro...
2   2  As soon as you have heard our letter, who(ever...
3   3  Send a copy of (this) letter of ours to every ...

Our submission format:
   id                                        translation
0   0  From the Kanesh colony to Aur-nya: The Kanesh ...
1   1  To the tablet of Ali-ahum, at the disposal of ...
2   2  gap> gap> gap> gap> gap> gap> gap> gap> gap> gap>
3   3  Me-pni seized us against Kanar and Wabar-rtim,...


In [9]:
# Save submission to ./submission.csv (root directory)
submission_path = Path("submission.csv")
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved to {submission_path}")
print(f"\nSubmission shape: {submission_df.shape}")

Submission saved to submission.csv

Submission shape: (4, 2)


## 5.7 Verify Submission

In [10]:
# Verify submission file
verification_df = pd.read_csv(submission_path)
print("=== Submission Verification ===")
print(f"Rows: {len(verification_df)}")
print(f"Columns: {verification_df.columns.tolist()}")
print(f"Missing translations: {verification_df['translation'].isna().sum()}")
print(f"Empty translations: {(verification_df['translation'].str.len() == 0).sum()}")

# Show first few entries
print("\nFirst few entries:")
print(verification_df.head())

=== Submission Verification ===
Rows: 4
Columns: ['id', 'translation']
Missing translations: 0
Empty translations: 0

First few entries:
   id                                        translation
0   0  From the Kanesh colony to Aur-nya: The Kanesh ...
1   1  To the tablet of Ali-ahum, at the disposal of ...
2   2  gap> gap> gap> gap> gap> gap> gap> gap> gap> gap>
3   3  Me-pni seized us against Kanar and Wabar-rtim,...


In [11]:
# Save submission summary
submission_summary = {
    "submission_file": str(submission_path),
    "total_predictions": len(submission_df),
    "model_type": "Ensemble (Beam Voting) - Sequential Loading",
    "models_used": [m["name"] for m in available_models],
    "num_models": len(available_models),
    "beam_size": BEAM_SIZE,
    "total_candidates_per_sample": len(available_models) * BEAM_SIZE,
    "voting_strategy": "length_normalized_beam_scoring",
    "missing_translations": int(verification_df['translation'].isna().sum()),
    "empty_translations": int((verification_df['translation'].str.len() == 0).sum()),
}

with open(OUTPUT_DIR / "submission_summary.json", 'w') as f:
    json.dump(submission_summary, f, indent=2)

print("\nSubmission summary saved")
print(json.dumps(submission_summary, indent=2))


Submission summary saved
{
  "submission_file": "submission.csv",
  "total_predictions": 4,
  "model_used": "Ensemble (Beam Voting) - Sequential Loading",
  "models_used": [
    "t5_small_v1"
  ],
  "num_models": 1,
  "beam_size": 5,
  "total_candidates_per_sample": 5,
  "voting_strategy": "length_normalized_beam_scoring",
  "missing_translations": 0,
  "empty_translations": 0
}


## 5.8 Final GPU Memory Cleanup

In [12]:
# Final cleanup
print("Final GPU memory cleanup...")
clear_gpu_memory()

print("\nAll models processed successfully!")
print(f"Submission saved to: {submission_path}")

Final GPU memory cleanup...
GPU memory cleared. Allocated: 0.00 GB, Reserved: 0.00 GB

All models processed successfully!
Submission saved to: submission.csv


## 5.9 List All Output Files

In [13]:
# List all output files
print("=== All Output Files ===")

print("\nOutput directory:")
for f in OUTPUT_DIR.rglob("*"):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.relative_to(OUTPUT_DIR)}: {size_kb:.1f} KB")

print("\nRoot directory (submission.csv):")
for f in Path(".").glob("*.csv"):
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

=== All Output Files ===

Output directory:
  eda_summary.json: 0.6 KB
  submission_summary.json: 0.4 KB
  external_data/akkademia_train.en: 4380.3 KB
  external_data/publications_summary.csv: 53.8 KB
  external_data/akkademia_parallel_corpus.csv: 9944.2 KB
  external_data/akkademia_train.ak: 4748.0 KB
  external_data/ebl_dictionary_processed.csv: 1582.9 KB
  external_data/external_data_summary.json: 2.3 KB
  external_data/akkademia_valid.ak: 269.5 KB
  external_data/akkademia_valid.en: 246.1 KB
  external_data/oa_lexicon_processed.csv: 3432.5 KB
  pre_processed_data/train.corpus: 9906.6 KB
  pre_processed_data/processing_summary.json: 0.7 KB
  pre_processed_data/val.jsonl: 1297.6 KB
  pre_processed_data/test.source: 0.7 KB
  pre_processed_data/train.jsonl: 11493.3 KB
  pre_processed_data/val.corpus: 1121.3 KB
  pre_processed_data/val_split.csv: 2288.3 KB
  pre_processed_data/test_processed.csv: 1.6 KB
  pre_processed_data/train_split.csv: 20216.1 KB
  pre_processed_data/train_processe

In [ ]:
# Verify notebook is valid JSON
notebook_path = Path("notebooks/05_model_submission.ipynb")
if notebook_path.exists():
    with open(notebook_path, 'r') as f:
        nb = json.load(f)
    print(f"Notebook is valid JSON with {len(nb['cells'])} cells")